# Lilly v2 — reader pass-7c (harvest filter + CUDA cropper)

Scope: `docs/V2-BOUNDARIES.md`. Latin Bosnian only (č ć đ š ž). No Cyrillic.

Pass-7 collapsed (loss inf at step 20,082). Pass-7b stayed finite ~5 epochs
but the gate refused: 14,885 EasyOCR auto-crops drowned 1,294 human labels.
Harvest auto-crop also ran **3.4 hours on CPU** while the T4 sat idle because
`app/ocr.py` hardcoded `gpu=False`.

Pass-7c:

- Auto-crop confidence **≥ 0.7**, capped at **2× human** train rows
- Harvest cropper uses **CUDA** when present (Mac stays CPU)
- LR 3e-6, cosine + 5% warmup, grad-clip 1, 5 epochs
- `python -u` so loss / before-after appear in the Kaggle log
- Continue from pass-4 `lilly.pth`

Attach: `lilly-read-pass1`, `lilly-ocr-crops`, and if present `lilly-ocr-harvest`.

Real crops merge **before** synthetic. Silent synthetic-only is a failed launch.
Setup mistakes and training/gate failures both stop the kernel. Fix that failure, then relaunch.

In [ ]:
# 1. Stop here unless the machine is actually set up
import os, subprocess, sys, urllib.error, urllib.request
from pathlib import Path

import torch
assert torch.cuda.is_available(), (
    "No GPU. Right panel -> Session options -> Accelerator -> GPU, then Save & Run All again.")

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(torch.cuda.device_count(), "GPU(s) visible, using:", torch.cuda.get_device_name(0))

def reachable(url):
    try:
        urllib.request.urlopen(url, timeout=20).close()
    except urllib.error.HTTPError:
        pass
    except Exception as exc:
        raise SystemExit(
            f"Cannot reach {url} ({exc}). Right panel -> Session options -> Internet -> On.")

for host in ("https://github.com", "https://pypi.org", "https://huggingface.co"):
    reachable(host)
print("network ok")

def run(*cmd):
    print("$", " ".join(str(c) for c in cmd), flush=True)
    subprocess.run([str(c) for c in cmd], check=True)

In [ ]:
# 2. Get the Lilly code — into scratch, NOT /kaggle/working
# Everything under /kaggle/working becomes Output. Training copies tens of
# thousands of PNGs into data/ocr/train and valid, and with the clone there too,
# `kaggle kernels output` spent 172 s on PNGs and git objects and never reached
# the weights zip at all. Only the zips below belong in Output.
SCRATCH = Path("/kaggle/temp") if Path("/kaggle/temp").is_dir() else Path("/tmp")
CLONE = SCRATCH / "Lilly"
subprocess.run(["rm", "-rf", str(CLONE)], check=True)
os.chdir(SCRATCH)
run("git", "clone", "-q", "https://github.com/ssaaffaakk/Lilly.git")
assert (CLONE / "training" / "train_ocr.py").is_file(), "clone produced nothing"
os.chdir(CLONE)
print("working in", os.getcwd(), "— Output will hold only the zips")

In [ ]:
# 3. Install what we need (~2 min)
# Do NOT pip-install torch from requirements.txt — that file pins CPU wheels for
# the Mac. Kaggle already ships a GPU build; replacing it wastes time and can
# break CUDA.
NEEDED = ["easyocr", "opencv-python-headless", "pillow", "huggingface_hub"]
pins = {}
for line in Path("requirements.txt").read_text(encoding="utf-8").splitlines():
    line = line.split("#")[0].strip()
    if line.startswith("--"):
        continue
    if "==" in line:
        pins[line.split("==")[0].strip().lower()] = line
run(sys.executable, "-m", "pip", "install", "-q", *[pins.get(n, n) for n in NEEDED])

In [ ]:
# 3b. Prove the GPU can backprop before an hour is spent generating images
x = torch.randn(256, 256, device="cuda", requires_grad=True)
y = (x @ torch.randn(256, 256, device="cuda")).sum()
y.backward()
print(f"GPU backprop ok on {torch.cuda.get_device_name(0)}")
torch.cuda.empty_cache()

In [ ]:
# 4. Base reader weights from Hugging Face + pass-1 from the attached dataset
# Dataset may land as files, as read/lilly.pth, or as read.zip (`-r zip`).
import shutil, zipfile
run("python3", "scripts/fetch_models.py")
READ = Path("models/lilly/read")
base = READ / "latin_g2.pth"
assert base.is_file() and base.stat().st_size > 1_000_000, "fetch_models missing read weights"
net = READ / "user_network"
for needed in ("lilly.yaml", "lilly.py"):
    assert (net / needed).is_file(), f"fetch_models missing user_network/{needed}"

input_root = Path("/kaggle/input")
print("attached inputs:",
      [str(p.relative_to(input_root)) for p in input_root.rglob("*")][:40]
      if input_root.is_dir() else "NONE")

unpack = Path("/tmp/pass1-unpack")
if unpack.exists():
    shutil.rmtree(unpack)
unpack.mkdir()
if input_root.is_dir():
    for z in input_root.rglob("*.zip"):
        dest = unpack / z.stem
        dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        print("unpacked", z)

hits = []
for root in (input_root, unpack):
    if root.is_dir():
        hits.extend(root.rglob("lilly.pth"))
if not hits:
    raise SystemExit(
        "No pass-1 weights under /kaggle/input. Attach lilly-read-pass1 "
        "or relaunch: python3 scripts/kaggle_train.py ocr")
src = hits[0]
shutil.copy(src, READ / "lilly.pth")
un = src.parent / "user_network"
if un.is_dir():
    shutil.copytree(un, net, dirs_exist_ok=True)
for name in ("lilly.yaml", "lilly.py"):
    sidecar = src.parent / name
    if sidecar.is_file():
        shutil.copy(sidecar, net / name)
INIT = READ / "lilly.pth"
print(f"starting reader from {src} -> {INIT}")
assert INIT.is_file() and INIT.stat().st_size > 100_000
print("pass-7 will continue from", INIT)

In [ ]:
# 5. Real hand-labelled crops FIRST — prepare_ocr_data deletes syn* files.
# If synthetic is written first, merging real wipes the generator images
# and train collapses to the 1,294 Latin crops (Kaggle pass-3 v5, ~2 min).
# Real names are not syn*; generate --build-splits then keeps them.
crops_dir = Path("data/ocr/crops")
crops_dir.mkdir(parents=True, exist_ok=True)
copied = 0
for root in (input_root, unpack):
    if not root.is_dir():
        continue
    for png in root.rglob("*.png"):
        # Harvested full photos are larger scene shots; human crops live in
        # lilly-ocr-crops. Copy everything named like a crop, skip harvest/ later.
        if "harvest" in str(png).lower():
            continue
        dest = crops_dir / png.name
        if not dest.exists():
            shutil.copy(png, dest)
            copied += 1
print(f"copied {copied} real crop PNGs from lilly-ocr-crops -> {crops_dir}")

merged = False
for labels in (Path("data/ocr/crops2/labels-human.tsv"),
               Path("data/ocr/crops/labels-human.tsv")):
    if not labels.is_file():
        continue
    rows = labels.read_text(encoding="utf-8").splitlines()
    if not rows:
        continue
    img_name = rows[0].split("\t")[0]
    img_path = labels.parent / img_name
    if img_path.is_file():
        run("python3", "training/prepare_ocr_data.py", "--labels", str(labels))
        print(f"merged real crops from {labels}")
        merged = True
        break
if not merged:
    raise SystemExit(
        "Pass-7 needs real crop PNGs. Attach lilly-ocr-crops "
        "or relaunch: python3 scripts/kaggle_train.py ocr")
print("real Latin crops are in train/valid; synthetic next (keeps these)")

In [ ]:
# 5b. Synthetic Bosnian crops on top of the real ones
# --build-splits keeps files whose names are not syn* (the real crops).
# Fonts: repo has data/fonts/README only. generate_ocr_data.py falls back to
# /usr/share/fonts on Linux and exits with a clear error if none have č/đ.
run("python3", "data/scripts/generate_ocr_data.py", "--count", "50000",
    "--seed", "44", "--build-splits")
train_gt = Path("data/ocr/train/gt.txt")
valid_gt = Path("data/ocr/valid/gt.txt")
print("flat synthetic written; photo-style next")

In [ ]:
# 5c. Photo-style synthetic — the harvest's last step, on the GPU box
# generate_ocr_photos.py appends photo*.png to train/valid. A later re-run of
# generate_ocr_data.py only clears syn*, so these survive. Calibration needs
# EasyOCR; that is why this runs here and not on the 8 GB Mac.
run("python3", "data/scripts/generate_ocr_photos.py", "--count", "20000")
n_photo = len(list(Path("data/ocr/train").glob("photo*.png")))
print(f"photo-style train files: {n_photo}")
assert n_photo > 1000, f"photo-style generator wrote too few: {n_photo}"


In [ ]:
# 5d. Harvested Commons photos → train-only auto-crops (not the valid gate)
# EasyOCR labels are noisy. They must not land in valid, or the real-crop gate
# scores a machine against itself. Prefix auto* so they are not syn* and not
# the human crop names.
# Commons files are almost all JPEG (pass-7: 480 jpg / 4 png / 2 jpeg). A
# `*.png` glob would copy four files and train as if harvest were missing.
HARVEST_EXT = {".jpg", ".jpeg", ".png", ".webp"}
AUTO_MIN_CONF = 0.7
AUTO_CAP_MULT = 2
# #region agent log
print(f"harvest filters defined: AUTO_MIN_CONF={AUTO_MIN_CONF} AUTO_CAP_MULT={AUTO_CAP_MULT}", flush=True)
# #endregion
photos_in = Path("/tmp/harvest-photos")
photos_in.mkdir(parents=True, exist_ok=True)
n_hv = 0
harvest_attached = False
for root in (input_root, unpack):
    if not root.is_dir():
        continue
    if any("ocr-harvest" in p.as_posix() for p in root.rglob("*")):
        harvest_attached = True
    for photo in root.rglob("*"):
        if not photo.is_file() or photo.suffix.lower() not in HARVEST_EXT:
            continue
        path = photo.as_posix().lower()
        if "harvest" not in path and "lilly-ocr-harvest" not in path:
            continue
        dest = photos_in / photo.name
        if not dest.exists():
            shutil.copy(photo, dest)
            n_hv += 1
print(f"harvested scene photos attached: {n_hv}")
if harvest_attached and n_hv < 20:
    raise SystemExit(
        f"lilly-ocr-harvest is attached but only {n_hv} photos were copied "
        f"(need jpg/jpeg, not png-only). Refusing to train without them.")
if n_hv >= 20:
    auto_dir = Path("data/ocr/crops-auto")
    if auto_dir.exists():
        shutil.rmtree(auto_dir)
    run("python3", "training/prepare_ocr_data.py", "--photos", str(photos_in),
        "--crops", str(auto_dir))
    labels = auto_dir / "labels.tsv"
    train_dir = Path("data/ocr/train")
    human_n = 0
    for row in train_gt.read_text(encoding="utf-8").splitlines():
        if not row.strip():
            continue
        name = row.split("\t", 1)[0]
        if not name.startswith(("syn", "photo", "auto_")):
            human_n += 1
    cap = max(human_n * AUTO_CAP_MULT, 0)
    candidates = []
    skipped_conf = skipped_bad = 0
    if labels.is_file():
        for line in labels.read_text(encoding="utf-8").splitlines():
            parts = line.split("\t")
            if len(parts) < 2:
                continue
            src, text = auto_dir / parts[0], parts[1].strip()
            try:
                conf = float(parts[2]) if len(parts) > 2 else 0.0
            except ValueError:
                conf = 0.0
            if not src.is_file() or not text:
                skipped_bad += 1
                continue
            if conf < AUTO_MIN_CONF:
                skipped_conf += 1
                continue
            candidates.append((conf, src, text))
    candidates.sort(key=lambda r: -r[0])
    skipped_cap = max(len(candidates) - cap, 0)
    extra = []
    for _conf, src, text in candidates[:cap]:
        name = f"auto_{src.stem}.png"
        shutil.copy(src, train_dir / name)
        extra.append(f"{name}\t{text}")
    if extra:
        with train_gt.open("a", encoding="utf-8") as f:
            f.write("\n".join(extra) + "\n")
    print(
        f"auto-crops: {len(extra)} kept of {len(candidates) + skipped_conf} "
        f"(conf>={AUTO_MIN_CONF}, cap {cap} = {human_n} human ×{AUTO_CAP_MULT}; "
        f"dropped low-conf {skipped_conf}, over-cap {skipped_cap}, bad {skipped_bad})",
        flush=True)
else:
    print("no harvest photos attached — training on human crops + synthetic only")

In [ ]:
# 5e. Repeat human real train rows ×2 (not auto*, not syn*, not photo*)
lines = [l for l in train_gt.read_text(encoding="utf-8").splitlines() if l.strip()]
def is_human(row):
    name = row.split("\t", 1)[0]
    return not name.startswith("syn") and not name.startswith("photo") and not name.startswith("auto_")
human = [l for l in lines if is_human(l)]
other = [l for l in lines if not is_human(l)]
assert human, "no human-labelled crops in train — the merge did not stick"
REAL_REPEAT = 2
import random
mixed_lines = human * REAL_REPEAT + other
random.Random(44).shuffle(mixed_lines)
train_gt.write_text("\n".join(mixed_lines) + "\n", encoding="utf-8")
train_n = len(mixed_lines)
valid_n = sum(1 for _ in open(valid_gt, encoding="utf-8"))
print(f"train oversampled: {len(human)} human ×{REAL_REPEAT} + {len(other)} generated "
      f"= {train_n:,}  |  valid {valid_n:,} (not repeated)")
assert train_n > 8000, f"mix too thin: train {train_n}"
assert valid_n > 100, f"only {valid_n} valid crops"


In [ ]:
# 5f. Four real GPU training steps before the long run (~30 s)
# Caught the cuda/cpu CTCLoss bug that killed v1 at step 1 after 2 min of setup.
print("$ python3 training/train_ocr.py --quick-test", flush=True)
smoke = subprocess.run(["python3", "training/train_ocr.py", "--quick-test"], check=False)
assert smoke.returncode == 0, (
    f"GPU OCR training smoke failed (exit {smoke.returncode}) — fix before the long run")
print("GPU OCR training smoke ok")

In [ ]:
# 6. HEAVY TRAINING — pass-7c: 5 epochs, cosine LR, tight grad clip
# Gate is real-crop words up + syn no-regression (see train_ocr.py).
# --keep-trained writes inspection weights first. A non-zero exit fails this kernel.
#
# pass-7 (v2): 1e-5 flat LR, grad-clip 5, 7 epochs → loss inf at step 20082.
# pass-7b (v3): 3e-6 cosine, but 14,885 unfiltered auto-crops; gate refused
#               and train_ocr stdout never reached the Kaggle log (buffered).
# pass-7c (v4): same LR recipe + filtered auto-crops + unbuffered logs.
os.environ["LILLY_RUN_ID"] = "heavy-pass7c"
os.environ["PYTHONUNBUFFERED"] = "1"
TRAINED = Path("models/lilly/read-trained.pth")
cmd = ["python3", "-u", "training/train_ocr.py",
       "--epochs", "5", "--batch-size", "16",
       "--lr", "3e-6", "--grad-clip", "1.0", "--warmup-frac", "0.05",
       "--weights", str(INIT), "--keep-trained", str(TRAINED)]
print("$", " ".join(cmd), flush=True)
proc = subprocess.run(cmd, check=False)
assert TRAINED.is_file() and TRAINED.stat().st_size > 100_000, (
    f"training wrote no weights at all (exit {proc.returncode})")

run("zip", "-j", "/kaggle/working/lilly-read-trained.zip", str(TRAINED))
print(f"train_ocr exit {proc.returncode} — "
      f"lilly-read-trained.zip saved before any packaging")
assert proc.returncode == 0, (
    f"train_ocr failed (exit {proc.returncode}) — not packaging a refused reader")

In [ ]:
# 7. Package what the app loads — only when the install gate passed
READ = Path("models/lilly/read")
NET = READ / "user_network"
for needed in ("lilly.yaml", "lilly.py"):
    assert (NET / needed).is_file(), f"missing user_network/{needed}"

assert (READ / "lilly.pth").is_file(), "gate passed but lilly.pth missing"
run("zip", "-qr", "/kaggle/working/lilly-read.zip",
    "models/lilly/read/lilly.pth",
    "models/lilly/read/user_network/lilly.yaml",
    "models/lilly/read/user_network/lilly.py")
size = Path("/kaggle/working/lilly-read.zip").stat().st_size
assert size > 100_000, f"zip too small: {size}"
print(f"lilly-read.zip — {size / 1048576:.1f} MB")

out = sorted(Path("/kaggle/working").rglob("*"))
print(f"\nOutput holds {len(out)} entries:")
for p in out[:20]:
    print(f"  {p.relative_to('/kaggle/working')}  {p.stat().st_size / 1048576:.1f} MB")
assert len(out) < 50, f"Output has {len(out)} entries — the fetch will drown"


**If the kernel is COMPLETE and `lilly-read.zip` is in Output:** unzip it over the repo root.
Keep `latin_g2.pth` and any previous `lilly.pth` as backup.

**If the kernel is ERROR:** do not install anything. Read the log, fix that failure, relaunch.